# GeoNUSAF — dataset upload to Zenodo from KagglePushes the Kathmandu Valley LULC dataset from the Kaggle input mount straight to aZenodo draft record, so the transfer never touches a home connection.**Before running, in the notebook sidebar:**1. **Internet** → **On** (Settings panel; requires a phone-verified Kaggle account)2. **Add Input** → attach the dataset `sugam24/satellite-datasets`3. **Add-ons → Secrets** → add a secret named `ZENODO_TOKEN`   Get the token from Zenodo → *Settings* → *Applications* → *Personal access tokens* → *New token*,   with scopes **`deposit:write`** and **`deposit:actions`** ticked.4. Accelerator → **None** (this is pure I/O, a GPU is wasted here)**Before running, on the Zenodo record:** delete any partial or failed file entry in theFiles section. A leftover entry with the same filename will make the upload fail.Expected wall-clock: ~10-20 min to build the archive, ~5-15 min to transfer.

## 1. Configuration`DEPOSITION_ID` is the number in the record URL: `https://zenodo.org/uploads/22349064`

In [ ]:
from dataclasses import dataclass, field@dataclassclass CFG:    # --- Zenodo ---    DEPOSITION_ID: str = "22349064"    SECRET_NAME:   str = "ZENODO_TOKEN"    ZENODO_BASE:   str = "https://zenodo.org"    # --- what to upload ---    ARCHIVE_NAME:  str = "kathmandu_valley_lulc_dataset.zip"    DATASET_DIR:   str = ""        # "" = auto-detect under /kaggle/input    TOP_FOLDER:    str = "satellite_dataset"   # folder name inside the zip    # --- staging ---    STAGE_DIR:     str = "/kaggle/temp"        # more room than /kaggle/working    COMPRESS:      bool = False    # PNGs are already compressed; store-only is ~20x faster    # --- transfer ---    MAX_ATTEMPTS:  int = 3    CHUNK:         int = 8 * 1024 * 1024    PROGRESS_EVERY_MB: int = 250    CONNECT_TIMEOUT: int = 30    READ_TIMEOUT:  int = 900    # --- safety ---    DELETE_REMOTE_PARTIALS: bool = True   # remove same-named file already on the draft    KEEP_ZIP_AFTER_UPLOAD:  bool = Falsecfg = CFG()print(f"record   : {cfg.ZENODO_BASE}/uploads/{cfg.DEPOSITION_ID}")print(f"archive  : {cfg.ARCHIVE_NAME}")

## 2. Imports and token

In [ ]:
import os, sys, json, time, hashlib, shutil, zipfile, subprocessfrom pathlib import Pathimport requeststry:    from kaggle_secrets import UserSecretsClient    TOKEN = UserSecretsClient().get_secret(cfg.SECRET_NAME)except Exception as e:    raise SystemExit(        f"Could not read secret '{cfg.SECRET_NAME}'.\n"        f"Add-ons -> Secrets -> add it, and tick the checkbox to attach it to this notebook.\n"        f"Underlying error: {e}"    )if not TOKEN or len(TOKEN) < 20:    raise SystemExit("Token looks empty or truncated. Re-copy it from Zenodo.")AUTH = {"Authorization": f"Bearer {TOKEN}"}print(f"token loaded ({len(TOKEN)} chars)")

## 3. Locate the dataset on the input mountSearches `/kaggle/input` for a folder holding both `images/` and `masks/`, so it keepsworking if the dataset slug changes.

In [ ]:
def find_dataset_root() -> Path:    if cfg.DATASET_DIR:        p = Path(cfg.DATASET_DIR)        if not p.exists():            raise SystemExit(f"DATASET_DIR does not exist: {p}")        return p    base = Path("/kaggle/input")    if not base.exists():        raise SystemExit("/kaggle/input not found - is a dataset attached to this notebook?")    for root, dirs, _ in os.walk(base):        d = set(dirs)        if "images" in d and "masks" in d:            return Path(root)    listing = "\n".join(f"  {p}" for p in sorted(base.glob("*")))    raise SystemExit(        "No folder containing both images/ and masks/ was found.\n"        f"Attached inputs:\n{listing}\n"        "Attach sugam24/satellite-datasets, or set cfg.DATASET_DIR by hand."    )SRC = find_dataset_root()n_img = len(list((SRC / "images").glob("*.png")))n_msk = len(list((SRC / "masks").glob("*.png")))src_bytes = sum(f.stat().st_size for f in SRC.rglob("*") if f.is_file())print(f"source   : {SRC}")print(f"images   : {n_img:,} png")print(f"masks    : {n_msk:,} png")print(f"size     : {src_bytes / 2**30:.2f} GiB")if n_img != n_msk:    print(f"\nWARNING: image/mask counts differ ({n_img} vs {n_msk}). Check before publishing.")if n_img == 0:    raise SystemExit("No PNGs found - wrong folder?")

## 4. Staging spaceThe archive is written to `/kaggle/temp`, which is scratch space and is not saved asnotebook output. `/kaggle/working` has a 20 GB output cap and is the wrong place for this.

In [ ]:
stage = Path(cfg.STAGE_DIR)stage.mkdir(parents=True, exist_ok=True)need = int(src_bytes * 1.02)free = shutil.disk_usage(stage).freeprint(f"staging  : {stage}")print(f"needed   : {need / 2**30:.2f} GiB")print(f"free     : {free / 2**30:.2f} GiB")if free < need:    alt = Path("/kaggle/working")    alt_free = shutil.disk_usage(alt).free    if alt_free >= need:        stage = alt        print(f"\nfalling back to {stage} ({alt_free / 2**30:.2f} GiB free)")    else:        raise SystemExit(            "Not enough scratch space to build the archive.\n"            "Zip images/ and masks/ as two separate uploads instead - "            "Zenodo allows up to 100 files per record."        )ZIP_PATH = stage / cfg.ARCHIVE_NAMEif ZIP_PATH.exists():    print(f"\nremoving stale archive from an earlier run: {ZIP_PATH}")    ZIP_PATH.unlink()

## 5. Build the archiveStore-only (no compression). PNG is already deflate-compressed, so compressing again buysmaybe 1-2% for roughly twenty times the CPU time.Paths inside the zip are `satellite_dataset/images/...` and `satellite_dataset/masks/...`,matching the layout described in the record.

In [ ]:
t0 = time.time()files = sorted(p for p in SRC.rglob("*") if p.is_file())total = len(files)mode = zipfile.ZIP_DEFLATED if cfg.COMPRESS else zipfile.ZIP_STOREDprint(f"writing {total:,} files -> {ZIP_PATH}")print(f"mode: {'deflate' if cfg.COMPRESS else 'store'}\n")written = 0with zipfile.ZipFile(ZIP_PATH, "w", compression=mode, allowZip64=True) as zf:    for i, f in enumerate(files, 1):        arc = Path(cfg.TOP_FOLDER) / f.relative_to(SRC)        zf.write(f, arcname=str(arc))        written += f.stat().st_size        if i % 200 == 0 or i == total:            el = time.time() - t0            rate = (written / 2**20) / max(el, 1e-9)            pct = 100 * i / total            print(f"\r  {i:,}/{total:,} ({pct:5.1f}%)  {written/2**30:6.2f} GiB  "                  f"{rate:6.1f} MiB/s  {el/60:5.1f} min", end="", flush=True)zip_bytes = ZIP_PATH.stat().st_sizeprint(f"\n\narchive  : {zip_bytes / 2**30:.2f} GiB in {(time.time()-t0)/60:.1f} min")

## 6. Verify the archive before spending bandwidth on itReads the central directory back and CRC-checks every member. This is the same check as`unzip -t`. Catching a bad archive here costs two minutes; catching it after publishingcosts a new DOI version.

In [ ]:
t0 = time.time()with zipfile.ZipFile(ZIP_PATH) as zf:    members = zf.namelist()    bad = zf.testzip()print(f"members  : {len(members):,}")print(f"expected : {total:,}")print(f"crc      : {'FAILED on ' + str(bad) if bad else 'all members OK'}")print(f"checked in {(time.time()-t0)/60:.1f} min")if bad or len(members) != total:    raise SystemExit("Archive is not sound. Delete it and rebuild before uploading.")

In [ ]:
t0 = time.time()h = hashlib.md5()with open(ZIP_PATH, "rb") as f:    while chunk := f.read(cfg.CHUNK):        h.update(chunk)LOCAL_MD5 = h.hexdigest()print(f"local md5: {LOCAL_MD5}  ({(time.time()-t0)/60:.1f} min)")

## 7. Connect to the recordZenodo runs two APIs side by side. The legacy deposit API exposes a single bucket URL youcan `PUT` a whole file to; the newer InvenioRDM API uses a three-stepinit / content / commit sequence. Which one serves a given draft depends on how it wascreated, so this tries legacy first and falls back.

In [ ]:
API_MODE = NoneBUCKET   = Nonelegacy_url = f"{cfg.ZENODO_BASE}/api/deposit/depositions/{cfg.DEPOSITION_ID}"r = requests.get(legacy_url, headers=AUTH, timeout=60)if r.status_code == 200 and r.json().get("links", {}).get("bucket"):    API_MODE = "legacy"    BUCKET   = r.json()["links"]["bucket"]    meta     = r.json().get("metadata", {})    print(f"api      : legacy deposit")    print(f"title    : {meta.get('title', '(untitled)')}")    print(f"bucket   : {BUCKET}")else:    draft_url = f"{cfg.ZENODO_BASE}/api/records/{cfg.DEPOSITION_ID}/draft"    r2 = requests.get(draft_url, headers=AUTH, timeout=60)    if r2.status_code == 200:        API_MODE = "rdm"        meta = r2.json().get("metadata", {})        print(f"api      : records draft (InvenioRDM)")        print(f"title    : {meta.get('title', '(untitled)')}")    else:        raise SystemExit(            f"Could not reach the draft.\n"            f"legacy -> {r.status_code}: {r.text[:300]}\n"            f"rdm    -> {r2.status_code}: {r2.text[:300]}\n\n"            "401/403 means the token lacks deposit:write, or belongs to a different account.\n"            "404 means the record id is wrong or the draft was deleted."        )

## 8. Clear any partial file already on the draftA cancelled browser upload can leave a half-written entry behind. Uploading the samefilename over it fails, usually with a message that does not say so.

In [ ]:
def list_remote():    if API_MODE == "legacy":        rr = requests.get(f"{legacy_url}/files", headers=AUTH, timeout=60)        return [(f["filename"], f.get("filesize")) for f in rr.json()] if rr.ok else []    rr = requests.get(f"{cfg.ZENODO_BASE}/api/records/{cfg.DEPOSITION_ID}/draft/files",                      headers=AUTH, timeout=60)    if not rr.ok:        return []    return [(e["key"], e.get("size")) for e in rr.json().get("entries", [])]remote = list_remote()print("files currently on the draft:")for k, s in remote or []:    size = f"{s/2**30:.2f} GiB" if s else "incomplete / no size"    print(f"  - {k}  ({size})")if not remote:    print("  (none)")if cfg.DELETE_REMOTE_PARTIALS:    for k, _ in remote:        if k == cfg.ARCHIVE_NAME:            if API_MODE == "legacy":                fid = [f["id"] for f in requests.get(f"{legacy_url}/files",                       headers=AUTH, timeout=60).json() if f["filename"] == k][0]                d = requests.delete(f"{legacy_url}/files/{fid}", headers=AUTH, timeout=120)            else:                d = requests.delete(                    f"{cfg.ZENODO_BASE}/api/records/{cfg.DEPOSITION_ID}/draft/files/{k}",                    headers=AUTH, timeout=120)            print(f"\ndeleted existing '{k}' -> {d.status_code}")

## 9. UploadOne streamed request. There is no resume on either API, so a dropped connection means thewhole file restarts — which is why this runs from a datacentre rather than a home link.

In [ ]:
class ProgressFile:    """File wrapper that reports throughput while requests streams it."""    def __init__(self, path, chunk, every_mb):        self.f     = open(path, "rb")        self.total = Path(path).stat().st_size        self.sent  = 0        self.chunk = chunk        self.mark  = every_mb * 2**20        self.next  = self.mark        self.t0    = time.time()    def __len__(self):        return self.total    def read(self, size=-1):        size = self.chunk if size in (-1, None) else min(size, self.chunk)        b = self.f.read(size)        self.sent += len(b)        if self.sent >= self.next or self.sent == self.total:            el   = time.time() - self.t0            rate = (self.sent / 2**20) / max(el, 1e-9)            pct  = 100 * self.sent / self.total            eta  = (self.total - self.sent) / 2**20 / max(rate, 1e-9) / 60            print(f"\r  {pct:5.1f}%  {self.sent/2**30:6.2f}/{self.total/2**30:.2f} GiB  "                  f"{rate:7.1f} MiB/s  eta {eta:5.1f} min", end="", flush=True)            self.next += self.mark        return b    def close(self):        self.f.close()def upload_once():    pf = ProgressFile(ZIP_PATH, cfg.CHUNK, cfg.PROGRESS_EVERY_MB)    hdrs = dict(AUTH)    hdrs["Content-Type"]   = "application/octet-stream"    hdrs["Content-Length"] = str(pf.total)    tmo = (cfg.CONNECT_TIMEOUT, cfg.READ_TIMEOUT)    try:        if API_MODE == "legacy":            return requests.put(f"{BUCKET}/{cfg.ARCHIVE_NAME}",                                data=pf, headers=hdrs, timeout=tmo)        base = f"{cfg.ZENODO_BASE}/api/records/{cfg.DEPOSITION_ID}/draft/files"        init = requests.post(base, headers={**AUTH, "Content-Type": "application/json"},                             data=json.dumps([{"key": cfg.ARCHIVE_NAME}]), timeout=60)        if init.status_code not in (200, 201):            print(f"\ninit failed {init.status_code}: {init.text[:300]}")            return init        put = requests.put(f"{base}/{cfg.ARCHIVE_NAME}/content",                           data=pf, headers=hdrs, timeout=tmo)        if put.status_code not in (200, 201):            return put        return requests.post(f"{base}/{cfg.ARCHIVE_NAME}/commit", headers=AUTH, timeout=300)    finally:        pf.close()resp = Nonefor attempt in range(1, cfg.MAX_ATTEMPTS + 1):    print(f"\nattempt {attempt}/{cfg.MAX_ATTEMPTS}  ({API_MODE})")    try:        resp = upload_once()        print(f"\n  -> HTTP {resp.status_code}")        if resp.status_code in (200, 201):            break        print(f"  {resp.text[:400]}")    except Exception as e:        print(f"\n  transfer failed: {type(e).__name__}: {e}")    if attempt < cfg.MAX_ATTEMPTS:        print("  retrying from the start in 30 s")        time.sleep(30)if resp is None or resp.status_code not in (200, 201):    raise SystemExit("Upload did not succeed. See the status and body above.")

## 10. Verify what landedCompares the size and MD5 Zenodo computed on its side against the local values. Matchingchecksums mean the bytes on Zenodo are the bytes you built.

In [ ]:
info = resp.json()if API_MODE == "rdm":    info = requests.get(        f"{cfg.ZENODO_BASE}/api/records/{cfg.DEPOSITION_ID}/draft/files/{cfg.ARCHIVE_NAME}",        headers=AUTH, timeout=60).json()remote_size = info.get("size") or info.get("filesize")remote_ck   = (info.get("checksum") or "").replace("md5:", "")print(f"local  size : {zip_bytes:,}")print(f"remote size : {remote_size:,}" if remote_size else "remote size : unknown")print(f"local  md5  : {LOCAL_MD5}")print(f"remote md5  : {remote_ck or 'not reported'}")ok_size = remote_size == zip_bytesok_ck   = (remote_ck == LOCAL_MD5) if remote_ck else Noneprint()print(f"size match  : {'YES' if ok_size else 'NO'}")print(f"md5 match   : {'YES' if ok_ck else ('NO' if ok_ck is False else 'not checkable')}")if not ok_size or ok_ck is False:    raise SystemExit("Mismatch - delete the file on the draft and run the upload again.")print(f"\nUpload verified.\n{cfg.ZENODO_BASE}/uploads/{cfg.DEPOSITION_ID}")

In [ ]:
if not cfg.KEEP_ZIP_AFTER_UPLOAD and ZIP_PATH.exists():    ZIP_PATH.unlink()    print(f"removed staging archive {ZIP_PATH}")

## 11. What to do next1. Refresh the record page. The Files validation warning should be gone and **Publish** available.2. If you have not already, click **Reserve DOI** so the paper can cite the dataset before publishing.3. Check the metadata one last time — **files become immutable once published.** Metadata stays   editable forever; changing a file after publication requires a new version with its own DOI.4. Publish.**If something failed**| Symptom | Cause ||---|---|| 401 / 403 on section 7 | Token missing `deposit:write`, or from a different Zenodo account || 404 on section 7 | Wrong record id, or the draft was deleted || 400 on upload | A file with this name is already on the draft — rerun section 8 || Secret not found | Add-on secret not attached to this notebook (tick its checkbox) || Connection errors throughout | Internet toggle is off in the notebook settings || Out of space in section 4 | Zip `images/` and `masks/` separately and upload as two files |